# Financial GraphRAG Engine — Demo Completo

Pipeline end-to-end: Ingesta → Grafo de Conocimiento → Recuperación Híbrida → Generación → Evaluación

**Stack:** Python 3.10+ · SEC EDGAR · bge-m3 · LanceDB · BM25 · Kùzu · Groq · LangChain · RAGAS

In [ ]:
from __future__ import annotations
import logging, sys, json, os
from pathlib import Path

logging.basicConfig(level=logging.WARNING, format="%(levelname)s | %(message)s")

os.chdir(Path.cwd().parent)
from src.pipeline import FinancialGraphRAGPipeline
from src.graph.communities import CommunityDetector

In [ ]:
# ── Configurar LLM (Groq u Ollama) ──
try:
    from langchain_groq import ChatGroq
    llm = ChatGroq(model="mixtral-8x7b-32768", temperature=0.0)
    print("Groq LLM ready")
except Exception as e:
    from langchain_community.chat_models import ChatOllama
    llm = ChatOllama(model="llama3.1", temperature=0.0)
    print(f"Groq no disponible, usando Ollama local ({e})")

---
## 1. Ingesta de un Informe 10-K

Descargamos, parseamos y chunkificamos un 10-K real desde SEC EDGAR.

In [ ]:
pipeline = FinancialGraphRAGPipeline(llm=llm)

chunks = pipeline.ingest_and_index(
    ticker="AAPL",
    year=2023,
)

print(f"Total chunks generados: {len(chunks)}")
if chunks:
    print(f"  Primer chunk [{chunks[0].section_id}]: {chunks[0].text[:120]}...")

### Estructura de un Chunk

In [ ]:
if chunks:
    c = chunks[0]
    print(f"chunk_id:       {c.chunk_id}")
    print(f"company_ticker: {c.company_ticker}")
    print(f"fiscal_year:    {c.fiscal_year}")
    print(f"section_id:     {c.section_id}")
    print(f"page_number:    {c.page_number}")
    print(f"token_count:    {c.token_count}")
    print(f"metadata:       {c.metadata}")

---
## 2. Grafo de Conocimiento

Los chunks ya se insertaron en Kùzu. Ahora inspeccionamos el grafo.

In [ ]:
conn = pipeline.graph.schema.connection

result = conn.execute("""
    MATCH (c:DocumentChunk) WHERE c.company_ticker = 'AAPL'
    RETURN c.chunk_id, c.section_id, c.fiscal_year
    LIMIT 5
""")
print("Chunks en Kùzu:")
while result.has_next():
    row = result.get_next()
    print(f"  {row[0][:16]}... | {row[1]} | FY{row[2]}")

In [ ]:
# ── Extraer tripletes con LLM ──
if chunks:
    triplets = pipeline.graph.extractor.extract_from_chunk(
        chunk_text=chunks[0].text[:2000],
        ticker=chunks[0].company_ticker,
        year=chunks[0].fiscal_year,
        section_id=chunks[0].section_id,
        chunk_id=chunks[0].chunk_id,
    )
    print(f"Tripletes extraídos: {len(triplets)}")
    for t in triplets[:3]:
        print(f"  ({t.source_name}) -[:{t.relation}]-> ({t.target_name})")

---
## 3. Recuperación Híbrida

### 3a. Búsqueda Vectorial (Dense)

In [ ]:
dense_results = pipeline.retrieval.dense.search(
    "What was Apple's revenue in 2023?", top_k=5
)
print(f"Resultados densos: {len(dense_results)}")
for r in dense_results[:3]:
    print(f"  score={r.score:.4f} | {r.text[:100]}...")

### 3b. Búsqueda Léxica (BM25)

In [ ]:
sparse_results = pipeline.retrieval.sparse.search(
    "risk factors competition supply chain", top_k=5
)
print(f"Resultados BM25: {len(sparse_results)}")
for r in sparse_results[:3]:
    print(f"  score={r.score:.4f} | {r.text[:100]}...")

### 3c. Recorrido en Grafo (Multi-Hop)

In [ ]:
graph_results = pipeline.retrieval.graph.search(
    "AAPL revenue segment performance", top_k=5
)
print(f"Resultados de grafo: {len(graph_results)}")
for r in graph_results[:3]:
    print(f"  path={r.traversal_path} | {r.text[:100]}...")

### 3d. RRF + Reranking

In [ ]:
fused = pipeline.retrieval.fusion.fuse(
    dense=dense_results,
    sparse=sparse_results,
    graph=graph_results,
    top_k=10,
)
print(f"Fusionados (RRF k=60): {len(fused)}")
for f in fused[:5]:
    print(f"  rrf={f.rrf_score:.4f} | d={f.dense_score} s={f.sparse_score} g={f.graph_score}")

reranked = pipeline.retrieval.reranker.rerank(
    query="What was Apple's revenue in 2023?",
    candidates=fused,
    top_k=5,
)
print(f"\nRerankeados: {len(reranked)}")
for r in reranked:
    print(f"  score={r.score:.4f} | {r.text[:80]}...")

---
## 4. Generación con Citas

Pipeline completo: pregunta → recuperación → RRF → rerank → generar respuesta con citas.

In [ ]:
result = pipeline.query("What was Apple's total net revenue for fiscal year 2023?")

print("=" * 70)
print("PREGUNTA:")
print(result.question)
print("\nRESPUESTA:")
print(result.answer)
print("\nCITAS:")
for c in result.citations[:3]:
    print(f"  [{c['company_ticker']} | FY{c['fiscal_year']} | {c['section_id']} | chunk: {c['chunk_id'][:12]}...]")
print(f"\nEstadísticas:")
print(f"  Dense:  {result.dense_results}")
print(f"  Sparse: {result.sparse_results}")
print(f"  Graph:  {result.graph_results}")
print(f"  Final:  {len(result.final_context)} chunks")

---
## 5. Comunidades de Grafo (Global Search)

Detección de comunidades via Leiden y resúmenes ejecutivos.

In [ ]:
detector = CommunityDetector(
    schema=pipeline.graph.schema,
    llm=llm,
)

communities = detector.detect_communities()
print(f"Comunidades detectadas: {len(communities)}")
for cid, members in sorted(communities.items(), key=lambda x: -len(x[1]))[:5]:
    print(f"  Comunidad {cid}: {len(members)} miembros — {members[:3]}...")

In [ ]:
summaries = detector.generate_summaries(communities, max_communities=3)
for s in summaries:
    print(f"\n{'='*60}")
    print(f"Comunidad {s.community_id} ({s.entity_count} entidades)")
    print(f"Top: {s.top_entities}")
    print(f"Resumen: {s.summary[:300]}...")

---
## 6. Evaluación con RAGAS

Corremos las 4 métricas sobre el dataset de prueba.

In [ ]:
from evals.run_ragas_eval import load_test_dataset, RagasEvaluator

samples = load_test_dataset("evals/test_dataset.json")
print(f"Muestras de evaluación: {len(samples)}")

evaluator = RagasEvaluator(pipeline=pipeline, llm=llm)
results = evaluator.run(samples=samples[:3], output_path="evals/results/demo_report.json")

In [ ]:
print("Scores individuales:")
for r in results:
    print(f"\n  Q: {r.sample.question[:60]}...")
    for metric, score in r.scores.items():
        print(f"    {metric:25s} {score:.4f}")

---
## Limpieza

In [ ]:
pipeline.close()
print("Pipeline cerrado.")